In [21]:
!pip install -qU langchain langchain_openai langchain-core>=1.0.0 langchain-community langgraph

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-elasticsearch 0.4.0 requires langchain-core<0.4.0,>=0.3.0, but you have langchain-core 1.1.1 which is incompatible.


In [22]:
%pip install -qU langchain-text-splitters langchain-elasticsearch langchain-classic psycopg[binary,pool]==3.2.6 psycopg2-binary

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
ERROR: Cannot install langchain-elasticsearch==0.3.0, langchain-elasticsearch==0.3.1, langchain-elasticsearch==0.3.2, langchain-elasticsearch==0.4.0 and langchain-text-splitters==0.0.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [4]:
!pip install pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 11.0 MB/s eta 0:00:00


In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [9]:
import os
from datetime import datetime
from typing import Dict, List, Optional, Literal
from pydantic import BaseModel, Field
import pandas as pd
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_elasticsearch import ElasticsearchStore

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
with open("/content/drive/MyDrive/api_key_openAI.txt") as archivo:
    apikey = archivo.read()
os.environ["OPENAI_API_KEY"] = apikey


In [12]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [13]:
from langchain.tools import tool

In [14]:
with open("/content/drive/MyDrive/postgrest.txt") as archivo:
  uribd = archivo.read()

In [15]:
from pydantic import BaseModel, Field
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Herramientas SQL

## a) tool_sql

es una herramienta de consulta a bases de datos muy útil, diseñada para responder preguntas sobre los datos de los estudiantes registrados en toda la universidad los cuales deben pasar todos por el examen medico.

In [16]:
from pydantic import BaseModel, Field
from langchain_community.utilities.sql_database import SQLDatabase


# Conexión a tu base de datos (ajusta la URI con tu motor y credenciales)
db_data = SQLDatabase.from_uri(uribd.replace('postgreSQL', 'postgresql'))

# Modelo de lenguaje
model = ChatOpenAI(verbose=True)

# Funciones auxiliares
def get_schema(_):
    return db_data.get_table_info()

def run_query(query):
    return db_data.run(query)

# Prompt para generar la consulta SQL
promptsql = ChatPromptTemplate.from_template("""
Basándonos en el esquema de la tabla siguiente, escribe una consulta SQL que responda a la pregunta del usuario:
    {schema}

    Pregunta: {question}
    Sql Query:
""")

# Cadena para generar la consulta SQL
sql_chain = (
    RunnablePassthrough.assign(schema=get_schema)
    | promptsql
    | model.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

# Prompt para convertir la respuesta SQL en lenguaje natural
promptsqlquery = ChatPromptTemplate.from_template("""
Basándonos en el esquema de la tabla inferior, pregunta, SQL Query y Respuesta, escribe una respuesta en lenguaje natural:
    {schema}

    Pregunta: {question}
    Sql Query: {query}
    SQL Respuesta: {response}
""")

# Cadena completa: genera la consulta, la ejecuta y devuelve respuesta natural
full_chain = (
    RunnablePassthrough.assign(query=sql_chain).assign(
        schema=get_schema,
        response=lambda vars: run_query(vars["query"]),
    )
    | promptsqlquery
    | model
)

# Define explicit input schema for the tool
class ToolInput(BaseModel):
    question: str = Field(description="La pregunta del usuario a la base de datos SQL.")

# Definición de la herramienta con esquema de entrada explícito
tool_sql = full_chain.with_types(input_type=ToolInput).as_tool(
    name="consulta_matriculas",
    description="Consulta en la base de datos información sobre estudiantes y matrículas"
)

/tmp/ipython-input-1260245195.py:60: LangChainBetaWarning: This API is in beta and may change in the future.
  tool_sql = full_chain.with_types(input_type=ToolInput).as_tool(


In [23]:
print(tool_sql.invoke({"question": "¿Cuántos estudiantes están matriculados en cada curso?"}).content)

Hay 2 estudiantes matriculados en Medicina Humana, 3 en Ingeniería Eléctrica, y 4 en Farmacia y Bioquímica, entre otros.


In [24]:
print(tool_sql.invoke({"question": "¿Cuál es el promedio de edad de los estudiantes?"}).content)

El promedio de edad de los estudiantes es de aproximadamente 24 años.


In [25]:
print(tool_sql.invoke({"question": "¿Cuántos estudiantes están matriculados en cada curso?"}).content)

Hay 2 estudiantes matriculados en Medicina Humana, 3 en Ingeniería Eléctrica, y 4 en Farmacia y Bioquímica, entre otros cursos.


In [26]:
print(tool_sql.invoke({"question": "¿Cuál es el promedio de edad de los estudiantes?"}))

content='El promedio de edad de los estudiantes es de aproximadamente 24 años.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 1131, 'total_tokens': 1147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CjGXeMRfyAhFwLTrWwOH9VUKfWwcZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019aec7f-b0e5-7c02-8623-7a9b594a9b30-0' usage_metadata={'input_tokens': 1131, 'output_tokens': 16, 'total_tokens': 1147, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## b) tool_evaluacionesmedicas

Este código crea una herramienta especializada para interactuar con la tabla evaluacionesmedicas de tu base de datos. Permite tanto consultar información existente como insertar nuevos registros de evaluaciones médicas.

In [31]:
from pydantic import BaseModel, Field
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize db_evaluaciones for the medical evaluations database
db_evaluaciones = SQLDatabase.from_uri(uribd.replace('postgreSQL', 'postgresql'))

# Define helper functions for the evaluacionesmedicas table
def get_schema_evaluaciones(_):
    # This assumes 'evaluacionesmedicas' is a table in the connected database
    return db_evaluaciones.get_table_info(table_names=["evaluacionesmedicas"])

def run_query_evaluaciones(query):
    return db_evaluaciones.run(query)

# Prompt to generate SQL query for evaluacionesmedicas with NOT NULL emphasis
promptsql_evaluaciones = ChatPromptTemplate.from_template(
    """
    Basándonos en el esquema de la tabla siguiente, escribe una consulta SQL que responda a la pregunta del usuario:
    {schema}

    La tabla relevante es 'evaluacionesmedicas'.

    Para sentencias INSERT en la tabla 'evaluacionesmedicas', asegúrate de incluir siempre valores para las columnas NOT NULL:
    'nombres', 'apellido_paterno', 'apellido_materno', 'dni_carnet_extranjeria', y 'nombre_escuela'.
    Debes extraer valores concretos para estas columnas directamente de la pregunta del usuario.
    NO generes valores de relleno (placeholders) como 'Nombre', 'ApellidoPaterno', 'DNI/Carnet de Extranjeria', etc.,
    si sus valores no se encuentran explícitamente en la pregunta del usuario.
    Si un valor NOT NULL no está presente en la pregunta, la sentencia INSERT no debe generarse o debe omitir la inserción.

    Pregunta: {question}
    Sql Query:
    """
)

# Chain to generate the SQL query for evaluacionesmedicas
sql_chain_evaluaciones = (
    RunnablePassthrough.assign(schema=get_schema_evaluaciones)
    | promptsql_evaluaciones
    | model.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

# Prompt to convert SQL response to natural language for evaluacionesmedicas
promptsqlquery_evaluaciones = ChatPromptTemplate.from_template(
    """
    Basándonos en el esquema de la tabla inferior, la pregunta, la SQL Query y la Respuesta, escribe una respuesta en lenguaje natural:
    {schema}

    Pregunta: {question}
    Sql Query: {query}
    SQL Respuesta: {response}
    """
)

# Full chain: generates query, executes it, and returns natural language response for evaluacionesmedicas
full_chain_evaluaciones = (
    RunnablePassthrough.assign(query=sql_chain_evaluaciones).assign(
        schema=get_schema_evaluaciones,
        response=lambda vars: run_query_evaluaciones(vars["query"]),
    )
    | promptsqlquery_evaluaciones
    | model
)

# Define explicit input schema for the tool
class ToolInput_evaluaciones(BaseModel):
    question: str = Field(description="La pregunta del usuario a la base de datos de evaluaciones médicas.")

# Definition of the tool with explicit input schema for evaluacionesmedicas
tool_evaluacionesmedicas = full_chain_evaluaciones.with_types(input_type=ToolInput_evaluaciones).as_tool(
    name="consulta_evaluacionesmedicas",
    description="Consulta en la base de datos información sobre las evaluaciones médicas de los alumnos, incluyendo la creación de nuevos registros."
)


In [ ]:
print(tool_evaluacionesmedicas.invoke({"question": "Registrar los resultados del estudiante con còdigo 22190250. Su examen se realizó el 11 de diciembre de 2025. En el tópico se le realizó un chequeo general a cargo del doctor QUISPE. Los resultados de laboratorio mostraron hemograma normal y glucosa normal, bajo la supervisión de la licenciada TORRES. "}))

In [39]:
print(tool_evaluacionesmedicas.invoke({"question": "¿Cuáles son los resultados de la evaluación médica del estudiante con DNI 12345678?"}))

content='La evaluación médica del estudiante con DNI 12345678 muestra que se realizó un Chequeo General el día 10 de diciembre del 2025. Fue atendido por el profesional Juan Pérez en el tópico de Laboratorio donde los resultados de los exámenes de hemograma y glucosa fueron normales. Además, se menciona la profesional María Gómez en el laboratorio. No se realizaron exámenes de odontología, psicología ni rayos X de tórax. No hay observaciones adicionales registradas. La última actualización de la evaluación fue el 4 de diciembre del 2025.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 753, 'total_tokens': 900, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C

## c) Tool identificador
Crea una herramienta `tool_identificar_persona` que, utilizando la base de datos existente y el modelo de lenguaje, identifique a una persona por su código de matrícula o número de documento en la tabla `matriculas`. La herramienta debe saludar a la persona por su nombre completo y especificar su rol (Estudiante, Administrativo o Médico). Finalmente, prueba esta herramienta invocándola primero con un ejemplo de `codigo_matricula` y luego con un ejemplo de `numero_documento`, y resume su funcionalidad.

In [40]:
from pydantic import BaseModel, Field
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize db_data (assuming it's already defined and connected to the database)
# db_data = SQLDatabase.from_uri(uribd.replace('postgreSQL', 'postgresql')) # This line is already present earlier

# 1. Define a function to get the schema for the 'matriculas' table
def get_schema_identificacion(_):
    return db_data.get_table_info(table_names=["matriculas"])

# 2. Create a ChatPromptTemplate for SQL query generation
promptsql_identificacion = ChatPromptTemplate.from_template(
    """
    Basándonos en el esquema de la tabla siguiente, escribe una consulta SQL que responda a la pregunta del usuario:
    {schema}

    La tabla relevante es 'matriculas'.
    Tu objetivo es seleccionar 'nombres', 'apellido_paterno', 'apellido_materno', y 'rol' (si está disponible o inferible) de la tabla 'matriculas'.
    Filtra los resultados usando 'codigo_matricula' o 'numero_documento' según lo que se proporcione en la pregunta del usuario.
    Si la pregunta es sobre una persona en general, sin un identificador específico (código de matrícula o DNI), no generes una consulta SQL.

    Pregunta: {question}
    Sql Query:
    """
)

# 3. Create a chain for SQL generation
sql_chain_identificacion = (
    RunnablePassthrough.assign(schema=get_schema_identificacion)
    | promptsql_identificacion
    | model.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

# 4. Create a ChatPromptTemplate for natural language response
prompt_respuesta_identificacion = ChatPromptTemplate.from_template(
    """
    Basándonos en la información proporcionada, saluda a la persona identificada por su nombre completo.
    Si no se encontró ninguna persona, indica que no se pudo identificar a la persona con la información proporcionada.

    Esquema: {schema}
    Pregunta: {question}
    Sql Query: {query}
    Respuesta SQL: {response}

    Respuesta en lenguaje natural:
    """
)

# 5. Create the full chain
full_chain_identificacion = (
    RunnablePassthrough.assign(query=sql_chain_identificacion).assign(
        schema=get_schema_identificacion,
        response=lambda vars: run_query(vars["query"]),
    )
    | prompt_respuesta_identificacion
    | model
)

# 6. Define explicit input schema for the tool
class ToolInput_Identificacion(BaseModel):
    question: str = Field(description="El código ingresado o la pregunta textual del usuario para identificar a una persona por código de matrícula o número de documento.")

# 7. Define the tool
tool_identificar_persona = full_chain_identificacion.with_types(input_type=ToolInput_Identificacion).as_tool(
    name="identificar_persona",
    description="Identifica una persona por código de matrícula o número de documento, saluda por su nombre completo " # y especifica su rol.
)


In [41]:
print(tool_identificar_persona.invoke({"question": "Identifica a la persona con número de documento 12345678." }).content)

No se pudo identificar a la persona con el número de documento 12345678 en la base de datos proporcionada.


In [42]:
print(tool_identificar_persona.invoke({"question": "70702146" }).content)

Hola Dominica Lee Zuñiga Reyes, ¡bienvenido/a!


## Herramienta de Cronograma Médico

`tool_cronograma_medico`  permitirá consultar la tabla `CronogramaExamenesMedicos` a través de lenguaje natural, incluyendo la conexión a la base de datos, funciones para el esquema, prompts para SQL y lenguaje natural, y la cadena runnable completa.


In [43]:
from pydantic import BaseModel, Field
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Assuming db_data and model are already initialized in previous cells
# db_data = SQLDatabase.from_uri(uribd.replace('postgreSQL', 'postgresql'))
# model = ChatOpenAI(verbose=True)
# run_query is also assumed to be available from tool_sql definition

# Initialize db_evaluaciones for the medical evaluations database
db_cronograma = SQLDatabase.from_uri(uribd.replace('postgreSQL', 'postgresql'))


# 1. Define a function to get the schema for the 'CronogramaExamenesMedicos' table
def get_schema_cronograma(_):
    return db_cronograma.get_table_info(table_names=["cronogramaexamenesmedicos"])

# 2. Create a ChatPromptTemplate for SQL query generation
promptsql_cronograma = ChatPromptTemplate.from_template(
    """
    Basándonos en el esquema de la tabla siguiente, escribe una consulta SQL que responda a la pregunta del usuario:
    {schema}

    La tabla relevante es 'cronograma_examenes_medicos'.

    Pregunta: {question}
    Sql Query:
    """
)

# 3. Construct a runnable chain for SQL generation
sql_chain_cronograma = (
    RunnablePassthrough.assign(schema=get_schema_cronograma)
    | promptsql_cronograma
    | model.bind(stop=["\nSQLResult:"])
    | StrOutputParser()
)

# 4. Create a ChatPromptTemplate for natural language response
prompt_respuesta_cronograma = ChatPromptTemplate.from_template(
    """
    Basándonos en el esquema de la tabla inferior, la pregunta, la SQL Query y la Respuesta, escribe una respuesta en lenguaje natural:
    {schema}

    Pregunta: {question}
    Sql Query: {query}
    SQL Respuesta: {response}

    Respuesta en lenguaje natural:
    """
)

# 5. Build the full_chain_cronograma runnable chain
full_chain_cronograma = (
    RunnablePassthrough.assign(query=sql_chain_cronograma).assign(
        schema=get_schema_cronograma,
        response=lambda vars: run_query(vars["query"]),
    )
    | prompt_respuesta_cronograma
    | model
)

# 6. Define a Pydantic BaseModel for the tool input
class ToolInput_Cronograma(BaseModel):
    question: str = Field(description="La pregunta del usuario a la base de datos del cronograma de exámenes médicos.")

# 7. Create the tool_cronograma_medico
tool_cronograma_medico = full_chain_cronograma.with_types(input_type=ToolInput_Cronograma).as_tool(
    name="consulta_cronograma_medico",
    description="Consulta en la base de datos información sobre el cronograma de exámenes médicos."
)

print("Tool 'tool_cronograma_medico' defined successfully.")

Tool 'tool_cronograma_medico' defined successfully.


In [44]:
print(tool_cronograma_medico.invoke({"question": "¿Cuál es el cronograma de exámenes médicos para el mes de diciembre?"}).content)

El cronograma de exámenes médicos para el mes de diciembre no tiene ninguna fila, ya que todas las fechas de inicio de los exámenes en la tabla son en enero.


In [47]:
print(tool_cronograma_medico.invoke({"question": "que fechas son para la escuela de Química?"}).content)

Las fechas para la escuela de Química son el 22 y 23 de marzo de 2026.


# RAG

In [48]:
!pip install pypdf

In [55]:
# Rutas de archivos
RUTA_NORMATIVAS = "/content/drive/MyDrive/proy/Normativa"
RUTA_PROCEDIMIENTOS = "/content/drive/MyDrive/proy/Procedimientos"



In [63]:
# Configuración Elasticsearch
ES_CONFIG = {
    "es_url": "http://34.28.18.170:9200",
    "es_user": "elastic",
    "es_password": "X1xW7QGK0nwCuDhyOX4V",
    "index_normativas": "medical_normativas",
    "index_procedimientos": "medical_procedimientos"
}

In [57]:
# Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Variables globales para los índices
vector_store_normativas = None
vector_store_procedimientos = None

# --- Función para cargar PDFs ---
def cargar_pdfs_directorio(directorio: str, categoria: str) -> List:
    documentos = []
    pdf_files = list(Path(directorio).glob("*.pdf"))

    print(f"\n Cargando {len(pdf_files)} archivos PDF desde: {directorio}")

    for pdf_path in pdf_files:
        try:
            loader = PyPDFLoader(str(pdf_path))
            docs = loader.load()

            for doc in docs:
                doc.metadata["categoria"] = categoria
                doc.metadata["archivo"] = pdf_path.name
                doc.metadata["ruta"] = str(pdf_path)
                doc.metadata["idioma"] = "es"
                doc.metadata["tipo_fuente"] = "PDF"

            documentos.extend(docs)
            print(f"   {pdf_path.name}: {len(docs)} páginas cargadas")

        except Exception as e:
            print(f"   Error leyendo {pdf_path.name}: {str(e)}")

    return documentos

In [52]:
# --- Inicializar base vectorial ---
def inicializar_base_vectorial(forzar_recarga: bool = False):
    global vector_store_normativas, vector_store_procedimientos

    print("\n=== INICIALIZANDO BASE VECTORIAL ===")

    # Cargar PDFs
    print("\n=== Cargando Normativas ===")
    docs_normativas = cargar_pdfs_directorio(RUTA_NORMATIVAS, "normativa")

    print("\n=== Cargando Procedimientos ===")
    docs_procedimientos = cargar_pdfs_directorio(RUTA_PROCEDIMIENTOS, "procedimiento")

    # Dividir documentos
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits_normativas = text_splitter.split_documents(docs_normativas)
    splits_procedimientos = text_splitter.split_documents(docs_procedimientos)

    print(f"\n Chunks normativas: {len(splits_normativas)}")
    print(f" Chunks procedimientos: {len(splits_procedimientos)}")

    # Crear índice Elasticsearch de Normativas
    print("\n Creando índice de NORMATIVAS...")
    vector_store_normativas = ElasticsearchStore.from_documents(
        documents=splits_normativas,
        embedding=embeddings,
        es_url=ES_CONFIG["es_url"],
        es_user=ES_CONFIG["es_user"],
        es_password=ES_CONFIG["es_password"],
        index_name=ES_CONFIG["index_normativas"],
    )
    vector_store_normativas.client.indices.refresh(index=ES_CONFIG["index_normativas"])
    print("Índice de normativas creado y actualizado")

    # Crear índice Elasticsearch de Procedimientos
    vector_store_procedimientos = ElasticsearchStore.from_documents(
        documents=splits_procedimientos,
        embedding=embeddings,
        es_url=ES_CONFIG["es_url"],
        es_user=ES_CONFIG["es_user"],
        es_password=ES_CONFIG["es_password"],
        index_name=ES_CONFIG["index_procedimientos"],
    )
    vector_store_procedimientos.client.indices.refresh(index=ES_CONFIG["index_procedimientos"])
    print("✓ Índice de procedimientos creado y actualizado")


In [49]:
# --- Buscar en documentos “motor de búsqueda”  ---
def buscar_en_documentos(pregunta: str, tipo_documento: str = "ambos", k: int = 3) -> List[Dict]:
    resultados = []
    if tipo_documento in ["normativa", "ambos"] and vector_store_normativas:
        docs = vector_store_normativas.similarity_search(pregunta, k=k)
        for doc in docs:
            resultados.append({
                "tipo": "normativa",
                "contenido": doc.page_content,
                "metadata": doc.metadata
            })
    if tipo_documento in ["procedimiento", "ambos"] and vector_store_procedimientos:
        docs = vector_store_procedimientos.similarity_search(pregunta, k=k)
        for doc in docs:
            resultados.append({
                "tipo": "procedimiento",
                "contenido": doc.page_content,
                "metadata": doc.metadata
            })
    return resultados

configruar el elastic

In [64]:
inicializar_base_vectorial()


=== INICIALIZANDO BASE VECTORIAL ===

=== Cargando Normativas ===

 Cargando 3 archivos PDF desde: /content/drive/MyDrive/proy/Normativa
   2001237-reglamento-de-examen-de-preseleccion-a-la-escuela-profesional-de-educacion-fisica-de-la-facultad-de-educacion-unmsm.pdf: 20 páginas cargadas
   Politicas de Confidencialidad.pdf: 4 páginas cargadas
   Reglamento_Alumnos_2025.pdf: 4 páginas cargadas

=== Cargando Procedimientos ===

 Cargando 6 archivos PDF desde: /content/drive/MyDrive/proy/Procedimientos
   FLUJOS DE ATENCIÓN topico.pdf: 2 páginas cargadas
   FLUJOS DE ATENCIÓN laboratorio.pdf: 4 páginas cargadas
   FLUJOS DE ATENCIÓN odontologia.pdf: 3 páginas cargadas
   FLUJOS DE ATENCIÓN PSICOLOGÍA.pdf: 3 páginas cargadas
   FLUJOS DE ATENCIÓN DE RAYOS X (TÓRAX).pdf: 4 páginas cargadas
   MANUAL COMPLETO DEL EXAMEN MÉDICO GENERAL.pdf: 7 páginas cargadas

 Chunks normativas: 70
 Chunks procedimientos: 37

 Creando índice de NORMATIVAS...
Índice de normativas creado y actualizad

In [59]:
resultados = buscar_en_documentos("¿Qué requisitos debo llevar al examen médico?", tipo_documento="ambos")
for r in resultados:
    print(r["tipo"], ":", r["contenido"][:200], "...")

In [60]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.schema.runnable import RunnablePassthrough
from langchain_classic.schema.output_parser import StrOutputParser


# retriever_chain usa buscar_en_documentos
retriever_chain = lambda question: buscar_en_documentos(question, tipo_documento="ambos")
modelo = ChatOpenAI(model="gpt-4.1-2025-04-14")

template = """
Eres un asistente para tareas de respuesta a preguntas de normativa y procedimientos.
Usa los siguientes fragmentos de contexto recuperado para responder la pregunta.
Si no sabes la respuesta, simplemente di que no lo sabes. Usa un máximo de tres oraciones y mantén la respuesta concisa.
Pregunta: {input}
Contexto: {context}
Respuesta:
"""
prompt = ChatPromptTemplate.from_template(template)

# Setup RAG pipeline
rag_chain = (
    {"context": retriever_chain,  "input": RunnablePassthrough()}
    | prompt
    | modelo
    | StrOutputParser()
)

# Modified format_docs to extract 'contenido' from dictionaries
def format_docs(docs):
    return "\n\n".join(doc["contenido"] for doc in docs)

qa_chain = (
    {
        "context": lambda question: format_docs(buscar_en_documentos(question, tipo_documento="ambos")),
        "question": RunnablePassthrough(),
    }
    | prompt
    | modelo
    | StrOutputParser()
)


In [65]:
print(rag_chain.invoke("¿Qué requisitos debo llevar al examen médico?"))

Para el examen médico debes llevar tu DNI, ropa casual, lapicero de tinta azul y mascarilla. Es importante que sigas estas indicaciones al momento de presentarte. No olvides cumplir con los horarios y procedimientos establecidos.


In [66]:
# create dataset
question = ["¿Qué requisitos debo llevar al examen médico?"]
response = []
contexts = []

# Inference
for query in question:
  response.append(rag_chain.invoke(query))
  # Corrected line to access 'contenido' key from the dictionary
  contexts.append([doc['contenido'] for doc in retriever_chain(query)])

# To dict
data = {
    "Pregunta": question,
    "Respuesta": response,
    "Contexto": contexts,
}

In [67]:
import pandas as pd
df = pd.DataFrame(data)
df

,Pregunta,Respuesta,Contexto
0,¿Qué requisitos debo llevar al examen médico?,"Para el examen médico debes llevar tu DNI, rop...",[UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS \nUn...


## integracion

In [68]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [86]:
# ============================================================================
# SISTEMA PROMPT POR ROL
# ============================================================================

SYSTEM_PROMPTS = {
    "estudiante": """Eres un asistente para ESTUDIANTES de la UNMSM.

PUEDES:
- Responder sobre requisitos del examen médico (usa RAG con rag_chain)
- Informar sobre cronogramas (usa tool_cronograma_medico)
- Explicar procedimientos (usa RAG con rag_chain)

NO PUEDES:
- Registrar evaluaciones médicas
- Ver reportes estadísticos completos

Si te piden algo que no puedes hacer, explica amablemente que es solo para médicos/administrativos.""",

    "medico": """Eres un asistente para MÉDICOS de la UNMSM.

PUEDES:
- Registrar resultados de exámenes (usa tool_evaluacionesmedicas)
  IMPORTANTE: Debes pedir TODOS estos datos antes de registrar:
  * Nombres, apellidos, DNI
  * Escuela y código de matrícula
  * Fecha del examen
  * Nombre del profesional responsable
  * Resultados específicos del área
- Consultar protocolos médicos (usa RAG con rag_chain)
- Ver estado de evaluaciones (usa tool_evaluacionesmedicas)
- Consultar datos de estudiantes (usa tool_sql)

Si faltan datos obligatorios, pídelos antes de registrar.""",

    "administrativo": """Eres un asistente para PERSONAL ADMINISTRATIVO de la UNMSM.

PUEDES:
- Generar reportes estadísticos (usa tool_sql)
- Consultar estado de exámenes por facultad/escuela (usa tool_evaluacionesmedicas y tool_sql)
- Ver normativa y procedimientos (usa RAG con rag_chain)
- Comparar evaluados vs total de estudiantes (cruza tool_sql con tool_evaluacionesmedicas)

Cuando generes reportes, sé específico con los números."""
}


In [87]:
# ============================================================================
# FUNCIÓN WRAPPER PARA RAG
# ============================================================================

@tool
def buscar_documentos_tool(pregunta: str) -> str:
    """Busca información en documentos de normativa y procedimientos médicos."""
    try:
        return rag_chain.invoke(pregunta)
    except Exception as e:
        return f"Error al buscar documentos: {str(e)}"

In [113]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

def crear_agente_por_rol(codigo_usuario: str):
    """Crea un agente configurado según el rol del usuario"""

    # 1. IDENTIFICAR USUARIO
    resultado_id = tool_identificar_persona.invoke({"question": codigo_usuario})
    identificacion = resultado_id.content

    # 2. EXTRAER ROL DE LA BD
    try:
        # Modificado: Usar '"Rol"' (con comillas dobles) para respetar la capitalización en PostgreSQL
        query = f"SELECT nombres, apellido_paterno, apellido_materno, codigo_matricula, \"Rol\" FROM matriculas WHERE codigo_matricula = '{codigo_usuario}' OR numero_documento = '{codigo_usuario}' LIMIT 1"
        print(f"Executing SQL Query: {query}") # Added print statement
        resultado_db = db_data.run(query)
        print(f"Raw DB Result: {resultado_db}") # New: Print raw database result

        if not resultado_db or "[]" in str(resultado_db):
            return None, "❌ Usuario no encontrado. Verifica tu código o DNI."

        # Parsear resultado
        import re
        # Modificado: Ajustar regex para manejar la lista que contiene la tupla
        match = re.search(r"\[\('([^']+)',\s*'([^']+)',\s*'([^']+)',\s*(\d+),\s*'([^']+)'\)\]", str(resultado_db))

        if match:
            nombres = match.group(1)
            ap_pat = match.group(2)
            ap_mat = match.group(3)
            codigo = match.group(4)
            rol = match.group(5)
            nombre_completo = f"{nombres} {ap_pat} {ap_mat}"
            rol = rol.lower()
        else:
            return None, "❌ Error al procesar datos del usuario."

    except Exception as e:
        return None, f"❌ Error de conexión: {str(e)}"

    # 3. SELECCIONAR HERRAMIENTAS SEGÚN ROL
    tools_por_rol = {
        "estudiante": [
            buscar_documentos_tool,
            tool_cronograma_medico
        ],
        "medico": [
            buscar_documentos_tool,
            tool_evaluacionesmedicas,
            tool_sql,
            tool_cronograma_medico
        ],
        "administrativo": [
            buscar_documentos_tool,
            tool_sql,
            tool_evaluacionesmedicas,
            tool_cronograma_medico
        ]
    }

    tools = tools_por_rol.get(rol, tools_por_rol["estudiante"])

    # 4. CREAR AGENTE CON SYSTEM PROMPT PERSONALIZADO
    system_prompt = f"""{SYSTEM_PROMPTS.get(rol, SYSTEM_PROMPTS["estudiante"])}

Usuario actual: {nombre_completo} - {codigo}
Rol: {rol.upper()}

IMPORTANTE: Usa las herramientas disponibles para responder. No inventes información."""

    model = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0)
    # FIXED: Bind the system_prompt to the model instead of using state_modifier
    model_with_system_prompt = model.bind(system_message=system_prompt)
    memory = MemorySaver()

    agente = create_react_agent(
        model=model_with_system_prompt, # Use the model with system message bound
        tools=tools,
        checkpointer=memory
    )

    # 5. MENSAJE DE BIENVENIDA
    mensaje_bienvenida = f"""¡Hola, {nombre_completo}! 👋

**Código:** {codigo}
**Rol:** {rol.upper()}

Bienvenido al Sistema de Exámenes Médicos de la UNMSM.

¿En qué te puedo ayudar hoy?"""

    return agente, mensaje_bienvenida, {"configurable": {"thread_id": codigo}}


In [114]:
# ============================================================================
# INTERFAZ SIMPLE
# ============================================================================

class SistemaExamenesMedicos:
    """Clase wrapper para facilitar el uso del sistema"""

    def __init__(self):
        self.agente = None
        self.config = None
        self.usuario = None
        self.rol = None

    def iniciar_sesion(self, codigo: str):
        """Inicia sesión con un código de usuario"""
        resultado = crear_agente_por_rol(codigo)

        if resultado[0] is None:
            return resultado[1]  # Mensaje de error

        self.agente, mensaje_bienvenida, self.config = resultado
        self.usuario = codigo

        # Extraer rol del mensaje
        if "ESTUDIANTE" in mensaje_bienvenida:
            self.rol = "estudiante"
        elif "MÉDICO" in mensaje_bienvenida or "MEDICO" in mensaje_bienvenida:
            self.rol = "medico"
        elif "ADMINISTRATIVO" in mensaje_bienvenida:
            self.rol = "administrativo"

        return mensaje_bienvenida

    def preguntar(self, mensaje: str):
        """Envía un mensaje al agente"""
        if not self.agente:
            return "⚠️ Debes iniciar sesión primero con tu código."

        try:
            response = self.agente.invoke(
                {"messages": [("user", mensaje)]},
                config=self.config
            )

            # Extraer última respuesta del agente
            ultima_respuesta = ""
            for msg in reversed(response["messages"]):
                if hasattr(msg, 'content') and msg.type == "ai":
                    ultima_respuesta = msg.content
                    break

            return ultima_respuesta if ultima_respuesta else "No obtuve respuesta del agente."

        except Exception as e:
            return f"⚠️ Error: {str(e)}"

    def cerrar_sesion(self):
        """Cierra la sesión actual"""
        self.agente = None
        self.config = None
        self.usuario = None
        self.rol = None
        return "👋 Sesión cerrada. ¡Hasta luego!"

In [115]:
# 1. Crear sistema
sistema = SistemaExamenesMedicos()

In [116]:
# 2. Iniciar sesión (identifica y crea agente con herramientas según rol)

# caso1: Código de estudiante
mensaje = sistema.iniciar_sesion("22030158")
mensaje

Executing SQL Query: SELECT nombres, apellido_paterno, apellido_materno, codigo_matricula, "Rol" FROM matriculas WHERE codigo_matricula = '22030158' OR numero_documento = '22030158' LIMIT 1
Raw DB Result: [('JESUS MANUEL', 'IBAÑEZ', 'GARCIA', 22030158, 'Estudiante')]


/tmp/ipython-input-1940063963.py:76: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


'¡Hola, JESUS MANUEL IBAÑEZ GARCIA! 👋\n\n**Código:** 22030158\n**Rol:** ESTUDIANTE\n\nBienvenido al Sistema de Exámenes Médicos de la UNMSM.\n\n¿En qué te puedo ayudar hoy?'

In [117]:
# 3. Hacer preguntas (el agente elige automáticamente la herramienta)
respuesta = sistema.preguntar("puedo ir el 1 de diciembre al examen?")

In [118]:
respuesta

'No puedes ir el 1 de diciembre al examen, ya que en la tabla de cronograma de exámenes médicos no hay ningún examen programado para esa fecha. Los exámenes programados son para los días 10 al 14 de enero, 15 al 17 de enero y 18 al 20 de enero del año 2026, todos pertenecientes a la escuela de Ciencias de la Salud.'

In [119]:
# 4. Cerrar sesión
sistema.cerrar_sesion()

'👋 Sesión cerrada. ¡Hasta luego!'

In [124]:
# 2. Iniciar sesión (identifica y crea agente con herramientas según rol)

# caso2: DNI de administrativo
mensaje = sistema.iniciar_sesion("56789012")
mensaje

Executing SQL Query: SELECT nombres, apellido_paterno, apellido_materno, codigo_matricula, "Rol" FROM matriculas WHERE codigo_matricula = '56789012' OR numero_documento = '56789012' LIMIT 1
Raw DB Result: [('MARÍA FERNANDA', 'GUTIERREZ', 'SALAZAR', 22222222, 'Administrativo')]


/tmp/ipython-input-1940063963.py:76: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


'¡Hola, MARÍA FERNANDA GUTIERREZ SALAZAR! 👋\n\n**Código:** 22222222\n**Rol:** ADMINISTRATIVO\n\nBienvenido al Sistema de Exámenes Médicos de la UNMSM.\n\n¿En qué te puedo ayudar hoy?'

In [125]:
# 3. Hacer preguntas (el agente elige automáticamente la herramienta)
respuesta = sistema.preguntar("que escuelas profesionales tienen al menos un alumno que haya pasado por el examen?")
respuesta

'La escuela profesional que tiene al menos un alumno que ha pasado por el examen y ha sido aprobado es **Ingeniería Eléctrica**.'

In [126]:
# 4. Cerrar sesión
sistema.cerrar_sesion()

'👋 Sesión cerrada. ¡Hasta luego!'

In [127]:
sistema = SistemaExamenesMedicos()

In [128]:
# 2. Iniciar sesión (identifica y crea agente con herramientas según rol)

# caso3: DNI de administrativo
mensaje = sistema.iniciar_sesion("63636363")
mensaje

Executing SQL Query: SELECT nombres, apellido_paterno, apellido_materno, codigo_matricula, "Rol" FROM matriculas WHERE codigo_matricula = '63636363' OR numero_documento = '63636363' LIMIT 1
Raw DB Result: [('MIGUEL ÁNGEL', 'QUISPE', 'RAMOS', 63636363, 'Médico')]


/tmp/ipython-input-1940063963.py:76: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(


'¡Hola, MIGUEL ÁNGEL QUISPE RAMOS! 👋\n\n**Código:** 63636363\n**Rol:** MÉDICO\n\nBienvenido al Sistema de Exámenes Médicos de la UNMSM.\n\n¿En qué te puedo ayudar hoy?'

In [129]:
# 3. Hacer preguntas (el agente elige automáticamente la herramienta)
respuesta = sistema.preguntar("cual es el estado de las evaluaciones??")
respuesta

'Las evaluaciones están en estado "Pendiente" para todas las escuelas y bloques listados en la tabla de cronograma de exámenes médicos. Esto significa que aún no han comenzado y se encuentran programadas para fechas futuras.'